# Setup

In [ ]:
!pip install strictyaml

In [ ]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib.pyplot as plt

import glob

import skimage as ski
from skimage import exposure, io, util, data, color, morphology, measure
from skimage.transform import hough_circle, hough_circle_peaks
from skimage.feature import canny
from skimage.draw import circle_perimeter
from skimage.util import img_as_ubyte
from skimage.filters import threshold_otsu, threshold_yen, threshold_local, gaussian, threshold_niblack, threshold_sauvola, sobel
from skimage.morphology import closing, footprint_rectangle, erosion, binary_opening, disk
from skimage.segmentation import watershed
from skimage.color import label2rgb
from skimage.measure import label
from scipy.ndimage import binary_fill_holes

from strictyaml import YAML, Map, Seq, Str, Float, Int, MapPattern
import os
import json

In [ ]:
input  = '/content/drive/MyDrive/Vision/Prepared/1_coins/'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# dynamic figure shortcuts - v1.3

def P(a=False, title='', size=2, axis=False, cmap='inferno', interpolation='bilinear', bins=False, fontsize=12, dpi=75):
    global FIG
    if 'FIG' not in globals():
        # first subplot
        FIG = plt.figure(figsize=(size, size), dpi=dpi)
        ax = FIG.add_subplot(1, 1, 1)
    else:
        # change the geometry and add a new subplot
        n = len(FIG.axes)
        FIG.set_figwidth(FIG.get_figheight() * (n + 1))
        gs = FIG.add_gridspec(1, n + 1)
        for i in range(n):
            FIG.axes[i].set_subplotspec(gs[i])
        ax = FIG.add_subplot(gs[-1])

    ax.axis(axis)
    if title: ax.set_title(title)
    if type(a) == bool: pass
    elif type(a) == np.ndarray and bins != False:
        ax.hist(a.ravel(), bins=bins)
        ax.set_aspect(np.diff(ax.get_xlim())[0] / np.diff(ax.get_ylim())[0])
    elif type(a) == np.ndarray and a.ndim == 2: ax.imshow(a, cmap=cmap, interpolation=interpolation)
    elif type(a) == np.ndarray and a.ndim == 3: ax.imshow(a, interpolation=interpolation)
    elif type(a) == str: ax.text(0, 0, a, fontsize=fontsize, fontfamily='monospace')
    else: ax.plot(a)

def S():
    global FIG
    if 'FIG' in globals(): plt.show(); del FIG

def V(*args, **kwargs):
    P(*args, **kwargs); S()

# Loading

In [ ]:
df = pd.read_csv(input + 'labels.csv', sep=';')


In [ ]:
int(df.loc[df["name"] == image_name, "real_count"].values[0])

In [ ]:
print(df[:10])

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.info

In [ ]:
files = list(sorted(glob.glob(input + '*.png')))

In [ ]:
files[:5]

Código "Original" do Skimage

In [ ]:
for file in files[:3]:
  # Load picture and detect edges
  image = io.imread(file)
  image = util.img_as_ubyte(image)
  edges = canny(image, sigma=3, low_threshold=10, high_threshold=50)


  # Detect two radii
  hough_radii = np.arange(20, 35, 2)
  hough_res = hough_circle(edges, hough_radii)

  # Select the most prominent 3 circles
  accums, cx, cy, radii = hough_circle_peaks(hough_res, hough_radii, total_num_peaks=3)

  # Draw them
  fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(10, 4))
  image = color.gray2rgb(image)
  for center_y, center_x, radius in zip(cy, cx, radii):
      circy, circx = circle_perimeter(center_y, center_x, radius, shape=image.shape)
      image[circy, circx] = (220, 20, 20)

  ax.imshow(image, cmap=plt.cm.gray)
  plt.show()

Código modificado, usando como base o original

# VER

https://scikit-image.org/docs/0.25.x/api/skimage.morphology.html#skimage.morphology.reconstruction

https://scikit-image.org/docs/0.25.x/auto_examples/features_detection/plot_holes_and_peaks.html#sphx-glr-auto-examples-features-detection-plot-holes-and-peaks-py

# TODO

- distância com base em fração da image

In [ ]:
from scipy.ndimage import binary_fill_holes

In [ ]:
results = []
for file in files[:5]:
  image = io.imread(file)
  image = util.img_as_ubyte(image) #[:400,:400]

  edges = canny(image, sigma=5, low_threshold=15, high_threshold=35)
  P(edges, 'edges', size=10, cmap='gray')

  edges_dilation = morphology.dilation(edges, morphology.disk(4))
  P(edges_dilation, 'dilation')

  filled = binary_fill_holes(edges_dilation)
  P(filled, 'filled')

  edges_erosion = morphology.erosion(filled, morphology.disk(14))
  P(edges_erosion, 'erosion')

  # Detect two radii
  hough_radii = np.arange(20, 45)
  hough_res = hough_circle(edges_erosion, hough_radii)

  height, width = image.shape[:2]

  min_xdistance = int(width * 0.08)
  min_ydistance = int(height * 0.08)

  if min_ydistance < min_xdistance:
    base = min_xdistance
  else:
    base = min_ydistance

  accums, cx, cy, radii = hough_circle_peaks(
      hough_res,
      hough_radii,
      min_xdistance=base,
      min_ydistance=base,
      threshold=0.7 * np.max(hough_res)
  )

  num_circulos = len(cx)

  image = color.gray2rgb(image)
  for center_y, center_x, radius in zip(cy, cx, radii):
    circy, circx = circle_perimeter(center_y, center_x, radius, shape=image.shape)
    image[circy, circx] = (220, 20, 20)

  P(image, f'final - {num_circulos} círculos detectados')
  S()

# Pensar em como sistematizar usando "grid search" dos hiperparâmetros

In [ ]:
# === CONFIGURAÇÕES ===
method_name = "HoughCircle"
yaml_path = "/content/drive/MyDrive/Vision/Results/hough_results.yaml"

parameters = {
    "sigma": 5,
    "low_threshold": 15,
    "high_threshold": 35,
    "dilation_disk": 4,
    "erosion_disk": 14,
    "radius_range": [20, 45],
    "distance_ratio": 0.08,
    "threshold_ratio": 0.7
}

results = []

linha = 0

for file in files:
  image = io.imread(file)
  image = util.img_as_ubyte(image) #[:400,:400]

  edges = canny(image, sigma=5, low_threshold=15, high_threshold=35)

  edges_dilation = morphology.dilation(edges, morphology.disk(4))

  filled = binary_fill_holes(edges_dilation)

  edges_erosion = morphology.erosion(filled, morphology.disk(14))

  # Detect two radii
  hough_radii = np.arange(20, 45)
  hough_res = hough_circle(edges_erosion, hough_radii)

  height, width = image.shape[:2]

  min_xdistance = int(width * 0.08)
  min_ydistance = int(height * 0.08)

  if min_ydistance < min_xdistance:
    base = min_xdistance
  else:
    base = min_ydistance

  accums, cx, cy, radii = hough_circle_peaks(
      hough_res,
      hough_radii,
      min_xdistance=base,
      min_ydistance=base,
      threshold=0.7 * np.max(hough_res)
  )

  num_circulos = len(cx)

  # Contagem real (do CSV)
  try:
      real_count = int(df[0])
  except IndexError:
      real_count = 0

  # Calcula acurácia
  accuracy = round(num_circulos / real_count, 3) if real_count > 0 else 0.0

  print(f"→ Detectado: {num_circulos}, Real: {real_count}, Acurácia: {accuracy}")

  # Armazena resultado
  results.append({
      "image": image_name,
      "real_count": real_count,
      "detected_count": num_circulos,
      "accuracy": accuracy
  })